## Data Cleaning and Analysis -- Problem Set 10: Data Cleaning, Regression, and Analytics
In this problem set, you will walk through the entire data cleaning pipeline, then run a regression analysis for a publically available dataset from https://data.cms.gov/provider-data/dataset/9n3s-kdb3. 

The `FY_2025_Hospital_Readmissions_Reduction_Program_Hospital.csv` dataset comes directly from the Centers for Medicare & Medicaid Services (CMS) Hospital Readmissions Reduction Program (HRRP). The HRRP aims to reduce avoidable hospital readmissions by financially incentivizing hospitals to improve the quality of care for certain high-risk conditions, including heart attack (AMI), heart failure (HF), pneumonia, chronic obstructive pulmonary disease (COPD), hip/knee replacement (THA/TKA), and coronary artery bypass graft surgery (CABG). Many of the conditions targeted by HRRP, such as COPD and heart failure, are sensitive to environmental exposures like air pollution, heat stress, and particulate matter (PM2.5). 

The dataset includes the following variables: 
- Facility Name: The full name of the hospital or healthcare facility
- Facility ID: The unique CMS identifier assigned to each facility
- State: The two-letter U.S. state or territory abbreviation where the facility is located
- Measure Name: The HRRP code identifying the condition or procedure being evaluated (EX. READM-30-HF-HRRP for 30-day heart failure readmissions)
- Number of Discharges: The total number of patient discharges for that condition during the reporting period
- Footnote: CMS notes or data quality flags that apply to the record
- Excess Readmission Ratio (ERR): The ratio of a hospital’s predicted to expected readmissions for a given condition. (ERR > 1 indicates more readmissions than expected and ERR < 1 indicates better-than-expected performance).
- Predicted Readmission Rate: The hospital’s estimated 30-day readmission rate (percentage), based on its patients’ characteristics and outcomes
- Expected Readmission Rate: The national benchmark readmission rate (percentage) expected for hospitals with similar patient case mixes
- Number of Readmissions: The observed number of readmissions at the facility for that condition
- Start Date: The beginning of the performance period for that measure (MM/DD/YYYY)
- End Date: The end of the performance period for that measure (MM/DD/YYYY)

For more information, check out the data dictionary for this dataset: https://data.cms.gov/provider-data/dataset/9n3s-kdb3#data-dictionary.

As always, use sources like Google, Stack Overflow, documentation, and other forums for help. Try to avoid AI in order to develop your own ability to conceptualize solution strategies.

## Setup & Data
Load required packages for data exploration and data visualization. 

Then, load the `FY_2025_Hospital_Readmissions_Reduction_Program_Hospital.csv` dataset as `cms_data` and view the first 10 rows.

In [0]:
library(tidyverse)
library(ggplot2)

cms_data <- read_csv("URL HERE")

head(cms_data, 10)


## Question 1: Cleaning Variables (8 points)

Real-world data, especially those released by public agencies, often come with quirks like inconsistent column names, mixed data types, and messy placeholders for missing values. Let's start the data cleaning process by creating a new, clean dataframe called `cms_clean` with the following modifications:

### Part A: Create consistent variable names (1 point)
Use the tidyverse function rename() to rename all variables so that they follow the snake case naming convention (eg. my_variable_name). 

OPTIONAL: trying out `clean_names()` from the `janitor` package (https://www.rdocumentation.org/packages/janitor/versions/1.2.0/topics/clean_names)


In [0]:
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# your grade is based on the following tests:

# Question 1A test 
if (sum(str_count(names(cms_clean),"_"))!=15|sum(!names(cms_clean)==names(tolower(cms_clean)))!=0){
    stop("You did not rename using snake case correctly.")
} else {
    print("Nice work, you correctly renamed your variables.")
}

### Part B: Convert date variables to their proper formats (1 point)

Next, inspect the structure of your data using `glimpse()`. Note that the date columns are stored as character strings.

Therefore, let's turn your `start_date` and `end_date` columns into a consistent date format (EX. YYYY-MM-DD)

In [0]:

# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# your grade is based on the following tests:

# Question 1B test 
if (!is.Date(cms_clean$start_date) | !is.Date(cms_clean$end_date)){
    stop("You did not convert your date variables correctly.")
} else {
    print("Nice work, you converted your date variables.")
}

### Part C: Clean and standardize missing data (6 points)
Next, take a closer look at the dataset and notice that several variables include text placeholders like "Too Few to Report" or "N/A" instead of leaving cells blank. While these labels make sense for human readers, R does not automatically interpret them as missing values. As a result, they are treated as text rather than true NAs, which can cause problems when calculating statistics or running regressions.

Modify your clean dataframe to do the following:

1. Replace all "N/A" entries with `NA_character_` in the following variables:
`number_of_discharges`, `excess_readmission_ratio`, `predicted_readmission_rate`, and `number_of_readmissions`. 
( Hint: Consider using `mutate()` with `across()` to apply your changes efficiently across multiple columns)

2. Replace "Too Few to Report" with 0 in the same variables. (This phrase indicates that the number of discharges or readmissions was too small for CMS to report publicly. Assigning a value of 0 preserves these records in the dataset while acknowledging that no measurable readmissions were reported)

3. Once placeholders have been standardized, convert the same variables into numeric data types so that R can correctly interpret them in summary statistics and regressions.

4. Use `summary()` or `glimpse()` to verify that your changes were applied correctly.

5. After cleaning, briefly explain why this approach is reasonable.
    - How does converting placeholders to NA or 0 improve the accuracy of your analysis?
    -  What assumptions are you making about hospitals labeled "Too Few to Report" when you treat those cases as zeros?



In [0]:
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# your grade is based on the following tests:

# Question 1C test 
if (cms_clean$number_of_discharges %>%is.na()%>% sum() != 10170|
cms_clean$number_of_discharges %>% mean(na.rm=T) %>% round(3) != 279.27|
cms_clean$excess_readmission_ratio %>%is.na()%>% sum() != 6583|
cms_clean$excess_readmission_ratio %>% mean(na.rm=T) %>% round(3)!= 1.002|
cms_clean$predicted_readmission_rate %>%is.na()%>% sum()!= 6583|
cms_clean$predicted_readmission_rate %>% mean(na.rm=T) %>% round(3)!= 14.995|
cms_clean$number_of_readmissions %>%is.na()%>% sum()!= 6583|
cms_clean$number_of_readmissions %>% mean(na.rm=T) %>% round(3)!= 32.734){
    stop("You did not convert your date variables correctly.")
} else {
    print("Nice work, you converted your date variables.")
}

YOUR ANSWER HERE

Now that the dataframe is clean, it’s important to start with some exploratory data analysis (EDA) to understand the relationships between variables in the dataset. 

Because each hospital appears in the dataset multiple times with one row for the hospital's performance for each condition (or measure), variables like predicted readmission rates and excess readmission ratios (ERR) are mathematically linked and do not provide independent information. Instead, we want to examine whether hospitals that perform poorly (or well) on one condition tend to show similar performance across other conditions, then whether average readmission performance differs systematically across states.


## Question 2:  Within-Hospital Correlation Across Measures (12 points)

First, we want to know whether hospitals that have high excess readmission ratios (ERR) for one condition tend to have high ERRs for other conditions?

### Part A: Creating a new dataframe (3 points)

Create a new dataframe called `cms_hosp_err` that:
- selects only the relevant variables: `facility_id`, `measure_name`, `excess_readmission_ratio`,
- drops rows with NA values for `excess_readmission_ratio`, and
- filters your data to include only facilities that report ERR for more than one measure. (Hint: use `group_by()` and `filter(n() > 1)`)

In [0]:
# YOUR CODE HERE
stop("No Answer Given!")


In [0]:
# your grade is based on the following tests:

# Question 2A test 
if (nrow(cms_hosp_err)!=11725|length(cms_hosp_err)!=3){
    stop("You did not create the dataframe correctly.")
} else {
    print("Nice work, you created the dataframe.")
}

### Part B: Summary Statistics (3 points)

Create a summary dataframe called `hosp_err_stats` that:
- computes the mean ERR per hospital (`mean_err`),
- the standard deviation in ERR per hospital (`sd_err`), and
- the number of measures per hospital (`n_measures`).

Use `head` to inspect this new dataframe.

In [0]:
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# your grade is based on the following tests:

# Question 2B test 
if (nrow(hosp_err_corr) != 2660 |
hosp_err_corr$mean_err %>% mean() %>% round(3) != 1.002 |
hosp_err_corr$sd_err %>% mean() %>% round(3) != 0.059 |
hosp_err_corr$n_measures %>% mean() %>% round(3)   != 4.408  ){
    stop("You did not create the correct summary dataframe.")
} else {
    print("Nice work, you created the summary dataframe.")
}

### Part C: Converting to wide-format (1 point)

Create a wide dataframe called `hosp_corr_wide` that reshapes `cms_hosp_err` into a wide format where each column represents a measure’s ERR.

In [0]:
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# your grade is based on the following tests:

# Question 2C test 
if (length(hosp_corr_wide) != 7 |
nrow(hosp_corr_wide) != 2660){
    stop("You did not pivot the dataframe correctly.")
} else {
    print("Nice work, you pivoted the dataframe.")
}

### Part D: Correlation Matrix (1 point)
Use the wide-format dataset `hosp_corr_wide` to calculate a correlation matrix of excess readmission ratios across conditions (for example, between heart failure and COPD). Store the matrix in an object called `cor_matrix_hosp`.

Apply the `cor()` function to all numeric columns while excluding the `facility_id` column. 

Then, view the matrix. (Hint: You can use `hosp_corr_wide[ , -1]` to remove the first column before computing the correlation matrix)

In [0]:
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# your grade is based on the following tests:

# Question 2D test 
if (as.vector(cor_matrix_hosp) %>% mean() %>% round(3) != 0.344){
    stop("You did not calculate the correlation matrix correctly.")
} else {
    print("Nice work, you calculated the correlation matrix.")
}

### Part E: Correlation Visualization (1 point)

Visualize the correlation matrix with `corrplot()`.

In [0]:
# YOUR CODE HERE
stop("No Answer Given!")

### Part F: Reflect (3 points)
Now, interpret your findings.
- Are certain measures highly correlated?
- What might this suggest about overall hospital quality?

YOUR ANSWER HERE

## Question 3: Differences in Excess Readmission Rates Across States (10 points)

In the previous analysis, you found weak correlations within a hospital between conditions, suggesting that hospital performance is not consistent across all measures. If overall hospital quality were the dominant driver of readmissions, we would expect strong positive correlations, where high-quality hospitals perform well across the board and low-quality hospitals perform poorly across all conditions. Therefore, performance differences might be driven less by overall hospital quality and more by the unique clinical risk factors, patient populations, and baseline readmission rates associated with each condition. 

However, while condition-specific factors might explain variation in excess readmission ratios (ERRs) within hospitals, there may still be broader geographic or systemic influences that shape hospital performance across states, such as differences in demographics, healthcare infrastructure, or regional policy environments. 

Now, we want to examine whether ERRs vary systematically across states after accounting for differences between conditions.

### Part A: Run a Regression with State Effects (3 points)

First, we want to run a regression where `excess_readmission_ratio` is the dependent variable and `state` is included as a categorical predictor.

Because each hospital reports multiple measures, and each measure represents a different condition with its own baseline risk and variability, we also include `measure_name` as a control variable. Some states may appear to have higher ERRs simply because they have more hospitals reporting conditions with inherently higher average readmission rates (such as chronic diseases). Therefore, controlling for `measure_name` allows us to isolate true geographic differences in performance, independent of which conditions are reported in each state.

1. Modify your `cms_clean` dataframe to ensure `state` and `measure_name` are factor variables. (If you try to include those variables as character strings directly in a regression (EX. lm(y ~ state)), R won’t know that each distinct string should represent a separate group with its own coefficient.)
2. Run the regression described above and name it `lm_state`.
3. Use `summary()` to view the regression results.

Note:
In R, when you run a linear model with categorical variables, one category from each factor is automatically chosen as the reference level. All other categories are compared against this reference. By default, lm() uses the first level of the factor (in alphabetical order) as the reference unless you explicitly set it with relevel(). Since state contains two-letter abbreviations, "AK" (Alaska) will be the reference because it appears first alphabetically.

In [0]:
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# your grade is based on the following tests:

# Question 3a test 
if (lm_state$coef %>% mean() %>% round(3)!=0.022 |
summary(lm_state)$coefficients[,2]%>% mean() %>% round(3) != 0.014){
    stop("You did not run the regression correctly.")
} else {
    print("Nice work, you ran the regression.")
}

### Part B: Coefficient Plot (3 points)

Now, we want to visualize each state’s coefficient relative to the reference state.

To make this plot, we’ll use the `broom` package to extract model results in a tidy format.
`broom::tidy()` takes a model object like `lm_state` and turns it into a data frame with one row per model term, including its coefficient, standard error, p-value, and confidence interval.

1. Execute the code below that:
- uses tidy() to extract the coefficients and 95% confidence intervals from your model,
- filters only the rows representing states (those whose term names start with "state"), and
- removes the "state" prefix from the term names so only the two-letter state codes appear in the plot.
2. Create a plot showing state effects on excess readmission ratios (ERRs)
    - sort the `state` variable by `estimate` (Hint: use `x = reorder(state, estimate)` within the `aes()`.)

In [0]:
# install.packages("broom")
library(broom)

coef_state <- tidy(lm_state, conf.int = TRUE) %>%
  filter(str_starts(term, "state")) %>%
  mutate(state = str_remove(term, "^state")) 

In [0]:
### create plot here
# YOUR CODE HERE
stop("No Answer Given!")

### Part C: Reflect (4 point)

Now, comment on your findings.
- Do the wide confidence intervals suggest high or low precision in the state-level estimates? What might explain this variability?
- Are there any states whose confidence intervals do not overlap zero, suggesting statistically significant differences compared to Alaska?
- What could explain why some states appear to perform better or worse than others, even after controlling for condition?

YOUR ANSWER HERE

---

## How to submit

When you've finished the problem set:

1. In the Colab menu bar, click **File → Save a copy in GitHub**.
2. In the dialog:
   - **Repository**: choose the repository GitHub Classroom created for you when you accepted this assignment (it will have your GitHub username in the name).
   - **Branch**: `main`.
   - **File path**: leave the filename as-is so it overwrites the starter notebook.
   - **Commit message**: something descriptive, e.g. *"Finished pset"*.
3. Click **OK**. Colab will push your completed notebook back to your assignment repository.

You can save to GitHub as often as you like — each save is just another commit. Your most recent commit before the due date is what gets graded.
